# LeetTrack Engine — Restructured

Modular version of the original notebook. Same DB schema, same revision/capacity
formulas — reorganized into small, reusable, testable functions.

Sections:
1. Setup & configuration
2. Database connection
3. Generic DB read/write helper
4. Revision math (pure functions)
5. Revision scheduling (DB query)
6. New-question selection (DB query)
7. Capacity & daily plan generation
8. Persisting progress (solve / revision) to MySQL
9. Display
10. Demo

## 1. Setup & configuration

In [1]:
import mysql.connector
from datetime import date, timedelta

# --- Tunable constants (all pulled from the spec, unchanged from the original) ---

DAILY_CAPACITY = 100

# Workload "cost" of one question of each difficulty, against DAILY_CAPACITY.
QUESTION_CAPACITY = {
    "Easy": 30,
    "Medium": 40,
    "Hard": 55
}

# Days until the *first* revision, per difficulty. Each later revision doubles
# the previous interval (see get_revision_interval below).
REVISION_BASELINE_DAYS = {
    "Easy": 7,
    "Medium": 5,
    "Hard": 3
}

# How many times a question can be revised in total, per difficulty.
REVISION_LIMITS = {
    "Easy": 2,
    "Medium": 4,
    "Hard": 5
}

## 2. Database connection

In [2]:
def get_connection():
    """Open a new MySQL connection to the leetcode_tracker database."""
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="ucs503",
        database="leetcode_tracker"
    )

## 3. Generic DB read/write helper

One function handles every query in this notebook, so cursor creation, parameter binding, `commit()`, and cursor cleanup are written exactly once.

In [3]:
def run_query(conn, query, params=None, fetch=False, many=False):
    """
    Execute one SQL statement against `conn`.

    fetch=True   -> SELECT: returns list[dict] (one dict per row).
    fetch=False  -> INSERT/UPDATE/DELETE: commits and returns affected row count.
    many=True    -> params is a list of tuples; uses executemany (bulk insert).

    Centralizing this means every read/write in the notebook persists correctly
    (commit is never forgotten) and every SELECT comes back as friendly dicts
    instead of raw tuples indexed by position.
    """
    cursor = conn.cursor(dictionary=True)
    try:
        if many:
            cursor.executemany(query, params or [])
        else:
            cursor.execute(query, params or ())

        if fetch:
            return cursor.fetchall()

        conn.commit()
        return cursor.rowcount
    finally:
        cursor.close()

## 4. Revision math (pure functions)

No DB access here on purpose — these are plain functions of `(difficulty, revision_counter, date)`, so the scheduling formula can be tested and reasoned about independently of MySQL.

In [4]:
def get_revision_interval(difficulty, revision_counter):
    """
    Days until the *next* revision, given how many revisions have already
    happened. The interval doubles every time:
        Easy:   7  -> 14 -> 28
        Medium: 5  -> 10 -> 20 -> 40
        Hard:   3  -> 6  -> 12 -> 24 -> 48
    """
    baseline = REVISION_BASELINE_DAYS[difficulty]
    return baseline * (2 ** revision_counter)


def can_be_revised(difficulty, revision_counter):
    """True if this question still has revisions left for its difficulty."""
    return revision_counter < REVISION_LIMITS[difficulty]


def get_next_revision_date(base_date, difficulty, revision_counter):
    """
    Due date for the next revision at this revision_counter, counted from
    base_date. Returns None once the difficulty's revision limit is reached
    (i.e. there is no "next" revision left).
    """
    if not can_be_revised(difficulty, revision_counter):
        return None
    return base_date + timedelta(days=get_revision_interval(difficulty, revision_counter))

In [ ]:
# Quick sanity check against the same cases the original notebook printed.
print(get_revision_interval("Easy", 0), get_revision_interval("Easy", 1), get_revision_interval("Easy", 2))
print(get_revision_interval("Medium", 0), get_revision_interval("Medium", 1))
print(get_revision_interval("Hard", 0), get_revision_interval("Hard", 1))

print(can_be_revised("Easy", 1), can_be_revised("Easy", 2))
print(can_be_revised("Hard", 4), can_be_revised("Hard", 5))

print(get_next_revision_date(date(2026, 8, 12), "Easy", 0))
print(get_next_revision_date(date(2026, 8, 12), "Hard", 5))  # limit reached -> None

## 5. Revision scheduling (DB query)

In [5]:
def get_due_revisions(conn, user_id, today):
    """
    This user's solved questions whose next revision is due (due_date <= today).

    Relies on due_date being NULL once a question's revision limit is used up
    (see get_next_revision_date), so exhausted questions are naturally excluded
    -- no need to repeat REVISION_LIMITS again inside the SQL.
    """
    return run_query(conn, """
        SELECT
            sq.question_id, sq.platform, qb.title, qb.difficulty, qb.topic, qb.url,
            sq.solved_date, sq.revision_counter, sq.last_revision_date, sq.due_date
        FROM solved_questions AS sq
        JOIN question_bank AS qb
            ON sq.question_id = qb.question_id AND sq.platform = qb.platform
        WHERE sq.user_id = %s
            AND sq.due_date IS NOT NULL
            AND sq.due_date <= %s
        ORDER BY sq.due_date ASC
    """, (user_id, today), fetch=True)

## 6. New-question selection (DB query)

In [6]:
def get_new_questions(conn, user_id, new_mix):
    """
    Pick up to `count` questions per difficulty that this user has not solved
    yet (no row in solved_questions). new_mix looks like
    {"Easy": 2, "Medium": 1, "Hard": 0}, as produced by find_best_new_question_mix.
    """
    questions = []
    for difficulty, count in new_mix.items():
        if count == 0:
            continue

        rows = run_query(conn, """
            SELECT qb.question_id, qb.platform, qb.title, qb.difficulty, qb.topic, qb.url
            FROM question_bank AS qb
            LEFT JOIN solved_questions AS sq
                ON qb.question_id = sq.question_id
                AND qb.platform = sq.platform
                AND sq.user_id = %s
            WHERE sq.question_id IS NULL
                AND qb.difficulty = %s
            ORDER BY qb.question_id
            LIMIT %s
        """, (user_id, difficulty, count), fetch=True)

        questions.extend(rows)

    return questions

## 7. Capacity & daily plan generation

In [7]:
def calculate_capacity_usage(questions):
    """Total capacity consumed by a list of question dicts (each has 'difficulty')."""
    return sum(QUESTION_CAPACITY[q["difficulty"]] for q in questions)


def find_best_new_question_mix(remaining_capacity):
    """
    Brute-force search over (Easy, Medium, Hard) counts for the combination that
    uses as much of remaining_capacity as possible without exceeding it.
    The search space is tiny (at most remaining_capacity // 30 per difficulty),
    so brute force is fine at this scale -- same approach as the original.
    """
    best_mix = {"Easy": 0, "Medium": 0, "Hard": 0}
    best_used = 0

    max_hard = remaining_capacity // QUESTION_CAPACITY["Hard"]
    max_medium = remaining_capacity // QUESTION_CAPACITY["Medium"]
    max_easy = remaining_capacity // QUESTION_CAPACITY["Easy"]

    for hard in range(max_hard + 1):
        for medium in range(max_medium + 1):
            for easy in range(max_easy + 1):
                used = (hard * QUESTION_CAPACITY["Hard"]
                        + medium * QUESTION_CAPACITY["Medium"]
                        + easy * QUESTION_CAPACITY["Easy"])
                if used <= remaining_capacity and used > best_used:
                    best_used = used
                    best_mix = {"Easy": easy, "Medium": medium, "Hard": hard}

    return best_mix, best_used


def create_daily_plan(conn, user_id, today):
    """
    Build today's plan for a user: due revisions always go in first (and consume
    capacity), then whatever capacity remains is filled with new questions.
    """
    revisions = get_due_revisions(conn, user_id, today)
    revision_capacity = calculate_capacity_usage(revisions)
    remaining_capacity = max(0, DAILY_CAPACITY - revision_capacity)

    new_mix, new_capacity = find_best_new_question_mix(remaining_capacity)
    new_questions = get_new_questions(conn, user_id, new_mix)

    return {
        "revisions": revisions,
        "new_questions": new_questions,
        "new_mix": new_mix,
        "revision_capacity": revision_capacity,
        "new_capacity": new_capacity,
        "remaining_capacity": remaining_capacity
    }

## 8. Persisting progress (solve / revision) to MySQL

The original had four overlapping functions here (`record_solved_question`, `add_solved_question`, `update_revision`, `complete_question`) that did the same insert/update in slightly different wrappers. One function covers both cases.

In [8]:
def record_question_progress(conn, user_id, question_id, platform, difficulty,
                               activity_date, revision_counter=None):
    """
    Persist a solve/revision event and return the resulting state.

    - revision_counter=None -> brand-new solve: INSERT a row with
      revision_counter=0 and the first due_date.
    - revision_counter=<int> -> a revision was just completed: UPDATE the row,
      incrementing the counter and recomputing due_date (None once the
      difficulty's revision limit is reached).
    """
    if revision_counter is None:
        next_counter = 0
        due_date = get_next_revision_date(activity_date, difficulty, next_counter)

        run_query(conn, """
            INSERT INTO solved_questions
                (user_id, question_id, platform, solved_date,
                 revision_counter, last_revision_date, due_date)
            VALUES (%s, %s, %s, %s, %s, %s, %s)
        """, (user_id, question_id, platform, activity_date,
              next_counter, activity_date, due_date))

        return {
            "type": "new_question",
            "revision_counter": next_counter,
            "last_revision_date": activity_date,
            "due_date": due_date
        }

    next_counter = revision_counter + 1
    due_date = get_next_revision_date(activity_date, difficulty, next_counter)

    run_query(conn, """
        UPDATE solved_questions
        SET revision_counter = %s, last_revision_date = %s, due_date = %s
        WHERE user_id = %s AND question_id = %s AND platform = %s
    """, (next_counter, activity_date, due_date, user_id, question_id, platform))

    return {
        "type": "revision",
        "revision_counter": next_counter,
        "last_revision_date": activity_date,
        "due_date": due_date,
        "completed": due_date is None
    }

## 9. Display

In [9]:
def display_daily_plan(daily_plan):
    """Pretty-print a daily plan dict (as returned by create_daily_plan)."""
    print("=" * 60)
    print("LEETTRACK DAILY PLAN")
    print("=" * 60)

    print("\nREVISIONS")
    print("-" * 60)
    revisions = daily_plan["revisions"]
    if not revisions:
        print("No revisions due today.")
    for i, q in enumerate(revisions, start=1):
        print(f"\n{i}. [{q['difficulty']}] {q['title']} (ID {q['question_id']})")
        print(f"   Topic: {q['topic']}   Revision #: {q['revision_counter']}   Due: {q['due_date']}")
        print(f"   {q['url']}")

    print("\nNEW QUESTIONS")
    print("-" * 60)
    new_questions = daily_plan["new_questions"]
    if not new_questions:
        print("No new questions available.")
    for i, q in enumerate(new_questions, start=1):
        print(f"\n{i}. [{q['difficulty']}] {q['title']} (ID {q['question_id']})")
        print(f"   Topic: {q['topic']}")
        print(f"   {q['url']}")

    used = daily_plan["revision_capacity"] + daily_plan["new_capacity"]
    print(f"\nCapacity used: {used}/{DAILY_CAPACITY}  "
          f"(revisions: {daily_plan['revision_capacity']}, new: {daily_plan['new_capacity']})")

## 10. Demo

Sets up a small, self-contained scenario (its own question IDs, so it won't collide with any real data already in the database) and walks through: new-question selection, revisions coming due, completing a revision, and the plan updating on the next call.

In [10]:
conn = get_connection()
print("Connected successfully!")

Connected successfully!


In [11]:
DEMO_USER_ID = 1  # reuses the existing demo user if present

DEMO_QUESTIONS = [
    (201, "leetcode", "Two Sum",                          "Easy",   "Array",          "https://leetcode.com/problems/two-sum/"),
    (202, "leetcode", "Valid Parentheses",                "Easy",   "Stack",          "https://leetcode.com/problems/valid-parentheses/"),
    (203, "leetcode", "3Sum",                             "Medium", "Two Pointers",   "https://leetcode.com/problems/3sum/"),
    (204, "leetcode", "Binary Tree Level Order Traversal", "Medium", "Tree",           "https://leetcode.com/problems/binary-tree-level-order-traversal/"),
    (205, "leetcode", "Word Ladder",                       "Hard",   "Graph",          "https://leetcode.com/problems/word-ladder/"),
    (206, "leetcode", "Minimum Window Substring",           "Hard",   "Sliding Window", "https://leetcode.com/problems/minimum-window-substring/"),
]
DEMO_IDS = [q[0] for q in DEMO_QUESTIONS]
placeholders = ",".join(["%s"] * len(DEMO_IDS))

# Idempotent setup: clear any leftovers from a previous run, then re-insert.
run_query(conn, f"DELETE FROM solved_questions WHERE question_id IN ({placeholders})", DEMO_IDS)
run_query(conn, f"DELETE FROM question_bank WHERE question_id IN ({placeholders})", DEMO_IDS)
run_query(conn, """
    INSERT INTO question_bank (question_id, platform, title, difficulty, topic, url)
    VALUES (%s, %s, %s, %s, %s, %s)
""", DEMO_QUESTIONS, many=True)
run_query(conn, "INSERT IGNORE INTO users (user_id, username) VALUES (%s, %s)", (DEMO_USER_ID, "demo_user"))

print("Demo question bank ready:", len(DEMO_QUESTIONS), "questions.")

Demo question bank ready: 6 questions.


### 10a. New questions being selected

Nothing is solved yet, so the whole daily plan should come from new questions.

In [12]:
demo_date = date(2026, 8, 12)

initial_plan = create_daily_plan(conn, DEMO_USER_ID, demo_date)
display_daily_plan(initial_plan)

LEETTRACK DAILY PLAN

REVISIONS
------------------------------------------------------------
No revisions due today.

NEW QUESTIONS
------------------------------------------------------------

1. [Easy] Two Sum (ID 201)
   Topic: Array
   https://leetcode.com/problems/two-sum/

2. [Easy] Valid Parentheses (ID 202)
   Topic: Stack
   https://leetcode.com/problems/valid-parentheses/

3. [Medium] 3Sum (ID 203)
   Topic: Two Pointers
   https://leetcode.com/problems/3sum/

Capacity used: 100/100  (revisions: 0, new: 100)


### 10b. Seed some solved questions at different revision stages

(Directly inserting mid-revision-counter rows here, purely to set up the scenario quickly -- exactly like the original notebook's demo did.)

In [13]:
solved_demo = [
    {"question_id": 201, "difficulty": "Easy",   "revision_counter": 0},
    {"question_id": 203, "difficulty": "Medium", "revision_counter": 1},
    {"question_id": 205, "difficulty": "Hard",   "revision_counter": 2},
]

for item in solved_demo:
    due_date = get_next_revision_date(demo_date, item["difficulty"], item["revision_counter"])
    run_query(conn, """
        INSERT INTO solved_questions
            (user_id, question_id, platform, solved_date, revision_counter, last_revision_date, due_date)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
    """, (DEMO_USER_ID, item["question_id"], "leetcode", demo_date,
          item["revision_counter"], demo_date, due_date))
    print(item["question_id"], item["difficulty"], "counter =", item["revision_counter"], "due =", due_date)

201 Easy counter = 0 due = 2026-08-19
203 Medium counter = 1 due = 2026-08-22
205 Hard counter = 2 due = 2026-08-24


### 10c. Revision questions being selected

On 2026-08-12 those due dates are all in the future, so no revisions should show up yet. Jumping to 2026-08-25, the Medium (due 08-22) and Hard (due 08-24) questions should now appear as due revisions.

In [14]:
print("--- Plan on 2026-08-12 (revisions not due yet) ---")
display_daily_plan(create_daily_plan(conn, DEMO_USER_ID, demo_date))

--- Plan on 2026-08-12 (revisions not due yet) ---
LEETTRACK DAILY PLAN

REVISIONS
------------------------------------------------------------
No revisions due today.

NEW QUESTIONS
------------------------------------------------------------

1. [Easy] Valid Parentheses (ID 202)
   Topic: Stack
   https://leetcode.com/problems/valid-parentheses/

2. [Medium] Binary Tree Level Order Traversal (ID 204)
   Topic: Tree
   https://leetcode.com/problems/binary-tree-level-order-traversal/

Capacity used: 100/100  (revisions: 0, new: 100)


In [15]:
test_date = date(2026, 8, 25)

print("--- Plan on 2026-08-25 (revisions now due) ---")
plan_with_revisions = create_daily_plan(conn, DEMO_USER_ID, test_date)
display_daily_plan(plan_with_revisions)

--- Plan on 2026-08-25 (revisions now due) ---
LEETTRACK DAILY PLAN

REVISIONS
------------------------------------------------------------

1. [Easy] Two Sum (ID 201)
   Topic: Array   Revision #: 0   Due: 2026-08-19
   https://leetcode.com/problems/two-sum/

2. [Medium] 3Sum (ID 203)
   Topic: Two Pointers   Revision #: 1   Due: 2026-08-22
   https://leetcode.com/problems/3sum/

3. [Hard] Word Ladder (ID 205)
   Topic: Graph   Revision #: 2   Due: 2026-08-24
   https://leetcode.com/problems/word-ladder/

NEW QUESTIONS
------------------------------------------------------------
No new questions available.

Capacity used: 125/100  (revisions: 125, new: 0)


### 10d. Completing a revision

Complete the revision for question 203 (Medium, currently at revision #1). We read the row before and after to show `revision_counter`, `last_revision_date`, and `due_date` actually changing in MySQL.

In [16]:
def fetch_solved_row(conn, user_id, question_id):
    rows = run_query(conn, """
        SELECT revision_counter, last_revision_date, due_date
        FROM solved_questions
        WHERE user_id = %s AND question_id = %s
    """, (user_id, question_id), fetch=True)
    return rows[0] if rows else None


print("Before:", fetch_solved_row(conn, DEMO_USER_ID, 203))

result = record_question_progress(
    conn, DEMO_USER_ID, 203, "leetcode", "Medium",
    activity_date=test_date, revision_counter=1
)
print("record_question_progress returned:", result)

print("After: ", fetch_solved_row(conn, DEMO_USER_ID, 203))

Before: {'revision_counter': 1, 'last_revision_date': datetime.date(2026, 8, 12), 'due_date': datetime.date(2026, 8, 22)}
record_question_progress returned: {'type': 'revision', 'revision_counter': 2, 'last_revision_date': datetime.date(2026, 8, 25), 'due_date': datetime.date(2026, 9, 14), 'completed': False}
After:  {'revision_counter': 2, 'last_revision_date': datetime.date(2026, 8, 25), 'due_date': datetime.date(2026, 9, 14)}


### 10e. Next daily plan reflecting the update

Question 203's due_date has moved to the future, so it should drop out of today's revision list -- only question 205 (Hard, still due 08-24) remains due.

In [17]:
print("--- Plan on 2026-08-25 again, after completing 203's revision ---")
updated_plan = create_daily_plan(conn, DEMO_USER_ID, test_date)
display_daily_plan(updated_plan)

--- Plan on 2026-08-25 again, after completing 203's revision ---
LEETTRACK DAILY PLAN

REVISIONS
------------------------------------------------------------

1. [Easy] Two Sum (ID 201)
   Topic: Array   Revision #: 0   Due: 2026-08-19
   https://leetcode.com/problems/two-sum/

2. [Hard] Word Ladder (ID 205)
   Topic: Graph   Revision #: 2   Due: 2026-08-24
   https://leetcode.com/problems/word-ladder/

NEW QUESTIONS
------------------------------------------------------------
No new questions available.

Capacity used: 85/100  (revisions: 85, new: 0)


In [18]:
conn.close()

2. Affordance of every query/function
"Affordance" here = what capability it actually unlocks for the app.
| Function                          | SQL Action     | Affordance                            | Called By             |
| --------------------------------- | -------------- | ------------------------------------- | --------------------- |
| `get_connection()`                | Connect MySQL  | Opens DB connection                   | Main program          |
| `fetch_users()`                   | `SELECT`       | Gets all users                        | Admin / batch         |
| `fetch_solved_questions()`        | `SELECT`       | Gets user's solve history             | Debugging             |
| `fetch_due_revisions()`           | `JOIN + WHERE` | Gets questions due today              | Revision planner      |
| `fetch_new_candidate_questions()` | `LEFT JOIN`    | Finds unsolved questions              | New-question planner  |
| `insert_question()`               | `INSERT`       | Adds question to bank                 | Seeding               |
| `insert_solved_question()`        | `INSERT`       | Records first solve + starts revision | `complete_question()` |
| `update_solved_question()`        | `UPDATE`       | Records revision + updates due date   | `complete_question()` |


3. Affordance of the pure (non-SQL) engine logic
These don't touch the DB — they're the math that decides what the queries above should even ask for.
| Function                        | Computes              | Affordance                     |
| ------------------------------- | --------------------- | ------------------------------ |
| `get_revision_interval()`       | Next revision gap     | Sets the next revision date    |
| `can_be_revised()`              | Checks revision limit | Decides if revision continues  |
| `complete_revision()`           | Next revision state   | Updates counter, date & status |
| `calculate_revision_capacity()` | Revision workload     | Calculates used daily capacity |
| `find_best_new_question_mix()`  | E/M/H combination     | Decides new questions to fetch |
